# NLLB-200: Kiswahili, Somali, Dholuo specialization — from our own Ekegusii checkpoint

**Scope, clarified:** English -> Kiswahili, English -> Somali, English -> Dholuo only,
PSA data only, one LoRA fine-tune per language - no ablation here, that already happened
in `02_nllb_training.ipynb`. What's different from an earlier version of this notebook:
this one initializes from **`Final_Training`'s own best-performing Ekegusii NLLB
checkpoint** (auto-selected in Section 3, by PSA-domain chrF2++), not from the raw
`facebook/nllb-200-distilled-600M` checkpoint. All three target languages already have
real, pretrained NLLB representations (confirmed against the tokenizer's actual vocabulary
in Section 4) - none of them need a new token or embedding initialization, so plain LoRA
is enough, same as the earlier version.

**Why the checkpoint choice matters, and what Section 5 does about it:** the winning
Ekegusii checkpoint was trained on `Final_Training`'s stage1/stage2/mixed data, which
itself draws heavily from `kenyan_psa_multilingual_dataset` - the same source file behind
most of this notebook's own PSA dataset (confirmed by exact-sentence matches, not just a
shared filename). That means the checkpoint's encoder has already seen some of these
English sentences before this notebook's fine-tuning even starts. Section 5 filters any
such overlap out of this dataset's **test** split specifically - train/val keep it, since
repeated exposure to the same input paired with a *different* target language isn't the
problem; a test set that's supposed to be held out but wasn't, is.

**Ordering note:** Sections 2-6 (verify language codes, load data, dedup, leakage-filter,
build records) have no dependency on `02_nllb_training.ipynb`'s ablation finishing and can
be run right now, while it's still in progress. Section 7 (select `BEST_NLLB_CHECKPOINT`)
is the one actual blocker - everything from there on needs a real result from that
ablation, not a guess.

## 1. Setup

In [8]:
!pip install -q torch transformers accelerate sentencepiece evaluate sacrebleu mlflow peft matplotlib pandas datasets


### GPU / CUDA check — run this before anything else

In [9]:
import torch
print(f"torch version       : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU visible to torch - fix this before continuing.")


torch version       : 2.11.0+cu128
CUDA available      : True
GPU                 : NVIDIA H100 PCIe


In [2]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")
if "final_training" not in str(PROJECT_ROOT).lower().replace(" ", "_"):
    print("\n  WARNING: 'Final_Training' doesn't appear in PROJECT_ROOT above - this notebook "
          "must live under Final_Training/Notebooks/ for find_project_root() to land in the "
          "right place. Move it there before continuing.")

from arch_config import ARCH_CONFIG, CHECKPOINTS_DIR, LOGS_DIR, DEPLOYMENT_DIR, DATA_DIR
import data_io, metrics, inference
from train import train_stage
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from datasets import Dataset
import pandas as pd

SEED = 42
set_seed(SEED)
MAX_LEN = 128
LORA_KWARGS = {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05}
OTHER_LANGS_DATA_DIR = Path(DATA_DIR) / "other_langs"

LANGUAGE_CODES = {"kiswahili": "swh_Latn", "somali": "som_Latn", "dholuo": "luo_Latn"}

# No tgt_code extension, no related_code, no embedding unfreezing - all three already have
# real NLLB representations (confirmed in Section 4).
OTHER_LANG_CFG = {
    "lora_target_modules": ["q_proj", "v_proj"],
    "freeze_layers": 6,
    "unfreeze_embeddings": False,
    "tgt_code": None,
    "related_code": None,
    "learning_rate": 5e-5,
    "batch_size": 32,
}
print("configured language codes (verified against the real tokenizer next):", LANGUAGE_CODES)


project root: /home/jovyan/PSA/Final_Training
configured language codes (verified against the real tokenizer next): {'kiswahili': 'swh_Latn', 'somali': 'som_Latn', 'dholuo': 'luo_Latn'}


## 2. Verify the language codes against the real NLLB tokenizer

Checked against the **raw base checkpoint** (`facebook/nllb-200-distilled-600M`), not
`BEST_NLLB_CHECKPOINT` - that's selected in Section 7, which is blocked until
`02_nllb_training.ipynb`'s ablation finishes, and doesn't need to be. NLLB-200's base
language coverage doesn't change across `Final_Training`'s fine-tunes - the only
difference any of them make to the tokenizer is adding the one `guz_Latn` token, which
has nothing to do with whether `swh_Latn`/`som_Latn`/`luo_Latn` are present. Everything
through Section 6 below runs with no dependency on the ablation - only Section 7 onward
is blocked.

In [3]:
# get_vocab(), not tokenizer.additional_special_tokens - that attribute raises
# AttributeError on this installed NllbTokenizer implementation (see peft_setup.py).
_check_tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
print("Checking each target language code against the real tokenizer:\n")
vocab = _check_tok.get_vocab()
verified = {}
for lang_name, code_guess in LANGUAGE_CODES.items():
    is_present = code_guess in vocab
    status = "CONFIRMED" if is_present else "NOT FOUND - do not trust this code"
    print(f"  {lang_name:12s} {code_guess:12s} {status}")
    verified[lang_name] = is_present
del _check_tok


Checking each target language code against the real tokenizer:

  kiswahili    swh_Latn     CONFIRMED
  somali       som_Latn     CONFIRMED
  dholuo       luo_Latn     CONFIRMED


## 3. Load the shared PSA dataset

Staged at `data/other_langs/Filtered_PSA_final.csv` inside `Final_Training` itself (not
referenced from outside it) - keeps this artifact a product of this project's own tree,
same reproducibility rule the rest of `Final_Training` follows.

In [4]:
DATASET_PATH = OTHER_LANGS_DATA_DIR / "Filtered_PSA_final.csv"
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"{DATASET_PATH} not found - upload it there first.")

psa_df = pd.read_csv(DATASET_PATH)
print(f"loaded {len(psa_df):,} rows")
print(psa_df["split"].value_counts().to_string())
print(f"\ncolumns: {list(psa_df.columns)}")
for col in ["English", "Kiswahili", "Somali", "Dholuo"]:
    blank = psa_df[col].isna() | (psa_df[col].astype(str).str.strip() == "")
    print(f"{col}: {blank.sum()} blank")


loaded 22,334 rows
split
train         17867
test           2234
validation     2233

columns: ['concept_id', 'English', 'Kiswahili', 'Somali', 'Dholuo', 'resource', 'dataset_origin', 'source_url', 'PSA_non_PSA', 'human_reviewed', 'split']
English: 0 blank
Kiswahili: 0 blank
Somali: 0 blank
Dholuo: 0 blank


## 4. Data quality: same-language mislabeling check

Already applied once when this file was built. Re-run here too as a live guard - if this
file is ever rebuilt/re-uploaded without that cleaning step, training won't silently
proceed on corrupted rows.

In [5]:
def flag_same_language(df, col_a, col_b):
    a = df[col_a].astype(str).str.strip().str.lower()
    b = df[col_b].astype(str).str.strip().str.lower()
    return a == b

def report_and_drop(df, mask, label):
    if mask.sum():
        print(f"{label}: dropping {mask.sum()} row(s) - English/Kiswahili identical")
        df = df[~mask].reset_index(drop=True)
    else:
        print(f"{label}: no same-language rows found - already clean")
    return df

psa_df = report_and_drop(psa_df, flag_same_language(psa_df, "English", "Kiswahili"), "psa_df")


psa_df: no same-language rows found - already clean


## 5. Remove train/test contamination against `Final_Training`'s PSA-domain data

Checked against `stage1`/`stage2`/`mixed` (training - `BEST_NLLB_CHECKPOINT` was directly
trained on these, in *both* directions: `Final_Training` is bidirectional English<->Ekegusii
and Kiswahili<->Ekegusii, so a row's `src` field is English text for one training example
and Kiswahili text for another - the loop below reads every row's `src` regardless of which,
so a pair passed to `Final_Training` on the Kiswahili side is already covered too) and
against `test_psa` (the checkpoint never trained on this, but `nllb_chrf_summary.csv`
Section 2 used it specifically to *pick* `BEST_NLLB_CHECKPOINT` - a sentence that did
double duty there and also ends up "held out" here would weaken this notebook's own test
claim). **Not** `test_bible`/`test_general` - this specialization is PSA-only, and there
are no Somali/Dholuo (or Kiswahili-as-target) pairs for bible/general content anywhere to
begin with, so checking against those two would just be checking a domain this dataset
never touches.

Only *this* dataset's **test** rows get filtered - train/val rows are left alone, since
repeated exposure to the same input paired with a different target language isn't the
problem; only the test split's validity is at stake.

In [6]:
curriculum_seen = data_io.load_curriculum(DATA_DIR)
already_seen = set()
for stage_name in ["stage1", "stage2", "mixed"]:
    for r in curriculum_seen[stage_name]:
        already_seen.add(r["src"].strip())
n_from_training = len(already_seen)

for r in data_io.load_jsonl(Path(DATA_DIR) / "test_psa.jsonl"):
    already_seen.add(r["src"].strip())
print(f"sentences seen via Final_Training's own training data: {n_from_training:,}")
print(f"additional sentences seen via Final_Training's own test_psa: {len(already_seen) - n_from_training:,}")
print(f"total distinct sentences checked against: {len(already_seen):,}")

test_mask = psa_df["split"] == "test"
leak_mask = test_mask & psa_df["English"].astype(str).str.strip().isin(already_seen)
print(f"\ntest rows already touched by Final_Training (train or test_psa): {leak_mask.sum()} of {test_mask.sum()}")
if leak_mask.sum():
    print("dropping those from the test split only:")
    psa_df = psa_df[~leak_mask].reset_index(drop=True)

print(f"\nremaining rows: {len(psa_df):,}")
print(psa_df["split"].value_counts().to_string())
del curriculum_seen, already_seen


sentences seen via Final_Training's own training data: 121,109
additional sentences seen via Final_Training's own test_psa: 3,019
total distinct sentences checked against: 124,128

test rows already touched by Final_Training (train or test_psa): 1569 of 2234
dropping those from the test split only:

remaining rows: 20,765
split
train         17867
validation     2233
test            665


## 6. Build training records for all three languages

Reads `psa_df["split"]` directly - does not re-split. Re-splitting here would discard the
file's own fixed split and, after Section 6's filtering, would no longer match row-for-row
across a re-run anyway.

In [7]:
def build_target_records(df, target_col, tgt_code, corpus_label):
    out = []
    for r in df.itertuples(index=False):
        tgt, en = str(getattr(r, target_col)).strip(), str(r.English).strip()
        if tgt and tgt.lower() != "nan" and en and en.lower() != "nan":
            out.append({"src_lang": "eng_Latn", "tgt_lang": tgt_code,
                        "src": en, "tgt": tgt, "corpus": corpus_label})
    return out

def split_records(target_col, tgt_code, corpus_label):
    train_rows = build_target_records(psa_df[psa_df["split"] == "train"], target_col, tgt_code, corpus_label)
    val_rows = build_target_records(psa_df[psa_df["split"] == "validation"], target_col, tgt_code, corpus_label)
    test_rows = build_target_records(psa_df[psa_df["split"] == "test"], target_col, tgt_code, corpus_label)
    print(f"{corpus_label}: train={len(train_rows):,} | val={len(val_rows):,} | test={len(test_rows):,}")
    return train_rows, val_rows, test_rows

kiswahili_train, kiswahili_val, kiswahili_test = split_records("Kiswahili", LANGUAGE_CODES["kiswahili"], "kiswahili_target")
somali_train, somali_val, somali_test = ([], [], [])
if verified.get("somali", False):
    somali_train, somali_val, somali_test = split_records("Somali", LANGUAGE_CODES["somali"], "somali_target")
dholuo_train, dholuo_val, dholuo_test = ([], [], [])
if verified.get("dholuo", False):
    dholuo_train, dholuo_val, dholuo_test = split_records("Dholuo", LANGUAGE_CODES["dholuo"], "dholuo_target")


kiswahili_target: train=17,867 | val=2,233 | test=665
somali_target: train=17,867 | val=2,233 | test=665
dholuo_target: train=17,867 | val=2,233 | test=665


## 7. Select the best-performing `Final_Training` NLLB checkpoint

This is the one point in the notebook that's actually blocked - everything above (Sections
2-6) has no dependency on `02_nllb_training.ipynb`'s ablation and can be run right now,
while that ablation is still finishing. This cell, and everything from here on, needs a
real winner from it - reads `02_nllb_training.ipynb` Section 7's own output, no guessing,
no hardcoded run name. PSA-domain chrF2++ is the deciding column: this project's actual
deployment target is Kenyan PSAs, not general Ekegusii fluency (same reasoning
`02_nllb_training.ipynb`'s own Section 9 uses for picking its demo run).

In [ ]:
summary_path = Path(LOGS_DIR) / "nllb_chrf_summary.csv"
if not summary_path.exists():
    raise FileNotFoundError(
        f"{summary_path} not found - run 02_nllb_training.ipynb's Section 7 (evaluation) "
        f"to completion first. This notebook needs a real winner from that ablation, not "
        f"a guess - there's no fallback checkpoint choice built in on purpose. Sections "
        f"2-6 above don't depend on this file and can be run in the meantime."
    )

chrf_summary = pd.read_csv(summary_path, index_col=0)
fine_tuned = chrf_summary.drop(index="zeroshot", errors="ignore")
print("Final_Training NLLB ablation results (chrF2++ per test set):")
print(fine_tuned.round(1).to_string())

best_run_name = fine_tuned["psa"].idxmax()
BEST_NLLB_CHECKPOINT = f"{CHECKPOINTS_DIR}/nllb_{best_run_name}/merged"
if not Path(BEST_NLLB_CHECKPOINT).exists():
    raise FileNotFoundError(
        f"{BEST_NLLB_CHECKPOINT} not found on disk - the results table names "
        f"'{best_run_name}' as the winner, but its checkpoint isn't there. Check "
        f"{CHECKPOINTS_DIR}/ directly before continuing."
    )
print(f"\nbest-performing Final_Training NLLB run (by PSA-domain chrF2++): {best_run_name}")
print(f"this notebook's specialization runs will initialize from: {BEST_NLLB_CHECKPOINT}")

# The real tokenizer, tied to BEST_NLLB_CHECKPOINT - Section 2's tokenizer was only ever
# a throwaway for the language-code check, this is the one Section 8 (tokenization) and
# every training/eval cell below actually use.
tokenizer = AutoTokenizer.from_pretrained(BEST_NLLB_CHECKPOINT)


## 8. Tokenization

In [ ]:
def nllb_preprocess(batch, tgt_lang):
    all_ids = []
    for sl, src, tgt in zip(batch["src_lang"], batch["src"], batch["tgt"]):
        tokenizer.src_lang = sl
        tokenizer.tgt_lang = tgt_lang
        enc = tokenizer(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
    return {"input_ids": [e["input_ids"] for e in all_ids],
            "attention_mask": [e["attention_mask"] for e in all_ids],
            "labels": [e["labels"] for e in all_ids]}

def tokenize_rows(rows, tgt_lang):
    if not rows:
        return None
    ds = Dataset.from_list(rows)
    return ds.map(lambda b: nllb_preprocess(b, tgt_lang), batched=True, batch_size=16)

tok_kiswahili_train = tokenize_rows(kiswahili_train, LANGUAGE_CODES["kiswahili"])
tok_kiswahili_val   = tokenize_rows(kiswahili_val, LANGUAGE_CODES["kiswahili"])
tok_kiswahili_test  = tokenize_rows(kiswahili_test, LANGUAGE_CODES["kiswahili"])

tok_somali_train = tokenize_rows(somali_train, LANGUAGE_CODES["somali"]) if somali_train else None
tok_somali_val   = tokenize_rows(somali_val, LANGUAGE_CODES["somali"]) if somali_val else None
tok_somali_test  = tokenize_rows(somali_test, LANGUAGE_CODES["somali"]) if somali_test else None

tok_dholuo_train = tokenize_rows(dholuo_train, LANGUAGE_CODES["dholuo"]) if dholuo_train else None
tok_dholuo_val   = tokenize_rows(dholuo_val, LANGUAGE_CODES["dholuo"]) if dholuo_val else None
tok_dholuo_test  = tokenize_rows(dholuo_test, LANGUAGE_CODES["dholuo"]) if dholuo_test else None
print("tokenization complete")


## 9. Train — layer-freeze vs. LoRA, per language, from `BEST_NLLB_CHECKPOINT`

Added ablation dimension (there wasn't one in the earlier version of this notebook):
each language gets both a `_lora_only` (no LoRA, falls back to `freeze_encoder_layers` -
see `02_nllb_training.ipynb`'s own note on what that name actually means in this
codebase) and a `_lora` (real LoRA adapter) run, mirroring the pairing `02_nllb_training.
ipynb` uses - except here it's a clean two-way comparison, not confounded with
embedding-unfreezing, since none of these three languages need a new token.

In [ ]:
_, kiswahili_lora_only_path, _ = train_stage(
    "nllb_kiswahili_lora_only", "nllb", OTHER_LANG_CFG, "kiswahili_target", BEST_NLLB_CHECKPOINT,
    tok_kiswahili_train, tok_kiswahili_val, tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=False, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
print(f"Kiswahili (layer-freeze) checkpoint: {kiswahili_lora_only_path}")


In [ ]:
_, kiswahili_lora_path, _ = train_stage(
    "nllb_kiswahili_lora", "nllb", OTHER_LANG_CFG, "kiswahili_target", BEST_NLLB_CHECKPOINT,
    tok_kiswahili_train, tok_kiswahili_val, tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
print(f"Kiswahili (LoRA) checkpoint: {kiswahili_lora_path}")


In [ ]:
somali_lora_only_path = somali_lora_path = None
if tok_somali_train is not None:
    _, somali_lora_only_path, _ = train_stage(
        "nllb_somali_lora_only", "nllb", OTHER_LANG_CFG, "somali_target", BEST_NLLB_CHECKPOINT,
        tok_somali_train, tok_somali_val, tokenizer, MAX_LEN, LORA_KWARGS,
        use_lora=False, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
    print(f"Somali (layer-freeze) checkpoint: {somali_lora_only_path}")
else:
    print("Somali training skipped - see Section 3.")


In [ ]:
if tok_somali_train is not None:
    _, somali_lora_path, _ = train_stage(
        "nllb_somali_lora", "nllb", OTHER_LANG_CFG, "somali_target", BEST_NLLB_CHECKPOINT,
        tok_somali_train, tok_somali_val, tokenizer, MAX_LEN, LORA_KWARGS,
        use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
    print(f"Somali (LoRA) checkpoint: {somali_lora_path}")
else:
    print("Somali training skipped - see Section 3.")


In [ ]:
dholuo_lora_only_path = dholuo_lora_path = None
if tok_dholuo_train is not None:
    _, dholuo_lora_only_path, _ = train_stage(
        "nllb_dholuo_lora_only", "nllb", OTHER_LANG_CFG, "dholuo_target", BEST_NLLB_CHECKPOINT,
        tok_dholuo_train, tok_dholuo_val, tokenizer, MAX_LEN, LORA_KWARGS,
        use_lora=False, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
    print(f"Dholuo (layer-freeze) checkpoint: {dholuo_lora_only_path}")
else:
    print("Dholuo training skipped - see Section 3.")


In [ ]:
if tok_dholuo_train is not None:
    _, dholuo_lora_path, _ = train_stage(
        "nllb_dholuo_lora", "nllb", OTHER_LANG_CFG, "dholuo_target", BEST_NLLB_CHECKPOINT,
        tok_dholuo_train, tok_dholuo_val, tokenizer, MAX_LEN, LORA_KWARGS,
        use_lora=True, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)
    print(f"Dholuo (LoRA) checkpoint: {dholuo_lora_path}")
else:
    print("Dholuo training skipped - see Section 3.")


In [ ]:
# One place collecting every trained checkpoint, keyed (language, variant) - Section 10
# (eval), 11 (deploy), and 12 (demo) all read from this rather than each re-deriving it.
runs = {
    ("kiswahili", "lora_only"): kiswahili_lora_only_path,
    ("kiswahili", "lora"): kiswahili_lora_path,
}
if somali_lora_only_path is not None:
    runs[("somali", "lora_only")] = somali_lora_only_path
    runs[("somali", "lora")] = somali_lora_path
if dholuo_lora_only_path is not None:
    runs[("dholuo", "lora_only")] = dholuo_lora_only_path
    runs[("dholuo", "lora")] = dholuo_lora_path

print(f"{len(runs)} checkpoint(s) trained:")
for (lang, variant), path in runs.items():
    print(f"  {lang:10s} {variant:10s} -> {path}")


## 10. Evaluate: zero-shot vs. every ablation variant, per language

Same structure `02_nllb_training.ipynb` Section 7 uses - one row per (run, test-set-
equivalent) combination, then a pivoted chrF2++ summary - just with "language" standing
in for that notebook's "test_set" dimension, since each of these three languages has its
own single (leakage-filtered, Section 6) test split rather than three shared ones.
Zero-shot uses the untouched `facebook/nllb-200-distilled-600M`, not
`BEST_NLLB_CHECKPOINT` - keeps "did PSA fine-tuning help at all" answerable on its own,
separate from "did starting from the Ekegusii checkpoint help or hurt."

In [ ]:
BASE_CHECKPOINT = "facebook/nllb-200-distilled-600M"
test_by_lang = {"kiswahili": kiswahili_test, "somali": somali_test, "dholuo": dholuo_test}

def load_and_generate(path, tgt_code):
    model = AutoModelForSeq2SeqLM.from_pretrained(path)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(path)
    def generate(text):
        tok.src_lang = "eng_Latn"
        enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = tok.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
        return tok.decode(out[0], skip_special_tokens=True)
    return model, tok, generate

eval_results = []
for lang, test_rows in test_by_lang.items():
    if not test_rows:
        continue
    model, tok, generate = load_and_generate(BASE_CHECKPOINT, LANGUAGE_CODES[lang])
    preds = [generate(r["src"]) for r in test_rows]
    score = metrics.score(preds, [r["tgt"] for r in test_rows], f"nllb_zeroshot_{lang}")
    score.update({"run": "zeroshot", "language": lang, "n": len(test_rows)})
    eval_results.append(score)
    del model
    torch.cuda.empty_cache()

for (lang, variant), path in runs.items():
    model, tok, generate = load_and_generate(path, LANGUAGE_CODES[lang])
    test_rows = test_by_lang[lang]
    preds = [generate(r["src"]) for r in test_rows]
    score = metrics.score(preds, [r["tgt"] for r in test_rows], f"nllb_{lang}_{variant}")
    score.update({"run": variant, "language": lang, "n": len(test_rows)})
    eval_results.append(score)
    del model
    torch.cuda.empty_cache()

results_df = pd.DataFrame(eval_results)[["run", "language", "n", "chrf2pp", "bleu"]]
results_df.to_csv(f"{LOGS_DIR}/nllb_other_languages_results.csv", index=False)
print(results_df.to_string(index=False))

print("\nchrF2++ summary, per run-variant x language:")
summary = results_df.pivot_table(index="run", columns="language", values="chrf2pp")
print(summary.round(1).to_string())
summary.to_csv(f"{LOGS_DIR}/nllb_other_languages_chrf_summary.csv")


## 11. Save deployment-ready checkpoints

In [ ]:
for (lang, variant), path in runs.items():
    inference.save_for_deployment(path, "nllb", f"nllb_{lang}_{variant}", deployment_dir=DEPLOYMENT_DIR)


## 12. Demonstration

Best variant per language auto-picked from Section 10's summary (highest chrF2++) - not
hardcoded, same reasoning as `BEST_NLLB_CHECKPOINT` in Section 2.

In [ ]:
samples = [
    "Farmers are urged to prioritize safe agrochemical usage this season.",
    "Ministry of Health: report suspected cholera cases immediately.",
]

for lang in test_by_lang:
    if lang not in summary.columns:
        continue
    best_variant = summary[lang].idxmax()
    demo_path = runs[(lang, best_variant)]
    demo_model = AutoModelForSeq2SeqLM.from_pretrained(demo_path)
    demo_model.to("cuda" if torch.cuda.is_available() else "cpu")
    demo_tok = AutoTokenizer.from_pretrained(demo_path)
    demo_tok.src_lang = "eng_Latn"
    forced_bos = demo_tok.convert_tokens_to_ids(LANGUAGE_CODES[lang])

    print(f"NLLB - English to {lang.capitalize()} (PSA-adapted, best variant: {best_variant})\n")
    for text in samples:
        enc = demo_tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(demo_model.device)
        out = demo_model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
        print(f"[en] {text}")
        print("  ->", demo_tok.decode(out[0], skip_special_tokens=True))
        print()
    del demo_model
    torch.cuda.empty_cache()
